# 🌾 Farm Fields – Boundary Visualisation & GeoJSON Export

This notebook loads images from the [Roboflow Farm Fields](https://universe.roboflow.com/odhiambb/farm-fields) dataset (YOLOv8 segmentation format), projects **polygon boundaries** on the images, and **exports them as GeoJSON** for [Kepler.gl](https://kepler.gl/demo).

**Dataset layout**
```
Farm Fields.v1i.yolov8/
├─ train/ (images/ + labels/)     43 images
├─ valid/                         13 images
├─ test/                           6 images
└─ data.yaml                       1 class: land-Sx1C
```

Each label `.txt` file contains YOLOv8 **segmentation** annotations:  
`<class_id> <x1> <y1> <x2> <y2> … <xN> <yN>` (normalised to [0, 1]).

In [ ]:
# ── Imports & setup ──────────────────────────────────────────
import os, sys, random, json
from pathlib import Path

import numpy as np
import matplotlib.pyplot as plt
from PIL import Image

%matplotlib inline
plt.rcParams.update({
    'figure.dpi': 120,
    'axes.facecolor': '#1a1a2e',
    'figure.facecolor': '#0f0f23',
    'text.color': 'white',
    'axes.labelcolor': 'white',
    'xtick.color': 'white',
    'ytick.color': 'white',
})

# Add repo root so we can import the helper modules
sys.path.insert(0, str(Path('.').resolve()))
from visualize_boundaries import (
    DATASET_ROOT, parse_yolo_seg_label, draw_boundaries,
    load_split, show_grid, show_single, dataset_summary,
    COLORS, EDGE_COLORS,
)

print(f'Dataset root: {DATASET_ROOT}')
print(f'Exists: {DATASET_ROOT.exists()}')

## 1️⃣ Dataset Overview

In [ ]:
dataset_summary()

## 2️⃣ Grid View – Training Split

Random sample of 9 training images with their **field boundaries** overlaid.

In [ ]:
show_grid('train', n=9, figsize=(18, 18))

## 3️⃣ Grid View – Validation Split

In [ ]:
show_grid('valid', n=6, figsize=(16, 12))

## 4️⃣ Grid View – Test Split

In [ ]:
show_grid('test', n=3, figsize=(16, 6))

## 5️⃣ Single Image – Side-by-Side (Original vs Boundaries)

Pick a specific image from any split and see it next to its annotated version.

In [ ]:
# Pick the first training image (change index to explore others)
pairs = load_split('train')
img_path, lbl_path = pairs[0]
print(f'Image: {Path(img_path).name}')
print(f'Label: {Path(lbl_path).name}')

show_single(img_path, lbl_path, figsize=(16, 7))

## 6️⃣ Detailed Polygon Explorer

Visualise a single image at full resolution with **numbered** polygons  
and a count/area table.

In [ ]:
from matplotlib.patches import Polygon as MplPolygon

# Pick an image with many polygons for a richer demo
idx = 0  # change me!
img_path, lbl_path = pairs[idx]

img = np.array(Image.open(img_path))
h, w = img.shape[:2]
anns = parse_yolo_seg_label(lbl_path, w, h)

fig, ax = plt.subplots(figsize=(14, 12))
ax.imshow(img)

print(f'{Path(img_path).name}  –  {len(anns)} polygons  |  {w}×{h} px\n')
print(f'{"#":>3s}  {"Vertices":>8s}  {"Area (px²)":>12s}  {"Area (%)":>8s}')
print('-' * 38)

for i, (cls_id, poly) in enumerate(anns):
    c  = COLORS[i % len(COLORS)]
    ec = EDGE_COLORS[i % len(EDGE_COLORS)]
    patch = MplPolygon(poly, closed=True, fill=True,
                       facecolor=c, edgecolor=ec, linewidth=2)
    ax.add_patch(patch)

    # Centroid label
    cx, cy = poly.mean(axis=0)
    ax.text(cx, cy, str(i), color='white', fontsize=7,
            fontweight='bold', ha='center', va='center',
            bbox=dict(boxstyle='round,pad=0.2', fc='black', alpha=0.65))

    # Area via Shoelace formula
    x, y = poly[:, 0], poly[:, 1]
    area = 0.5 * abs(np.dot(x, np.roll(y, 1)) - np.dot(y, np.roll(x, 1)))
    pct = area / (w * h) * 100
    print(f'{i:>3d}  {len(poly):>8d}  {area:>12,.0f}  {pct:>7.2f}%')

ax.set_title(f'{Path(img_path).name}  –  {len(anns)} field boundaries',
             fontsize=12, fontweight='bold', pad=8)
ax.axis('off')
plt.tight_layout()
plt.show()

## 7️⃣ Browse All Images

Change `SPLIT`, `START`, and `END` below to browse any range of images.  
Each image is shown side-by-side: **original** vs **with boundaries**.

In [ ]:
# ── Configuration ──────────────────────────────────────
SPLIT = 'train'   # 'train', 'valid', or 'test'
START = 0         # first image index (0-based)
END   = 5         # show images [START .. END-1]
# ──────────────────────────────────────────────────────

all_pairs = load_split(SPLIT)
print(f'{SPLIT} split: {len(all_pairs)} images total  |  showing [{START}..{END-1}]\n')

for i in range(START, min(END, len(all_pairs))):
    ip, lp = all_pairs[i]
    print(f'── Image {i}: {Path(ip).name}')
    show_single(ip, lp, figsize=(16, 7))

---

# 🗺️ GeoJSON Export for Kepler.gl

The images have **no embedded geospatial metadata** (Roboflow stripped EXIF).  
We assign each image a synthetic geographic tile over farmland near **Surat, India** and convert the pixel polygons to **lon/lat** coordinates.

| Parameter | Value |
|-----------|-------|
| Anchor point | 21.20°N, 72.78°E |
| Tile size | ~0.005° (~500 m) |
| Grid layout | 10 columns, images tiled left-to-right, top-to-bottom |

In [ ]:
## 8️⃣ Export boundaries to GeoJSON

from export_geojson import build_geojson, ANCHOR_LAT, ANCHOR_LON, TILE_SIZE_DEG

geojson = build_geojson(splits=['train', 'valid', 'test'], grid_cols=10)
n_features = len(geojson['features'])

output_path = Path('.').resolve() / 'farm_fields_boundaries.geojson'
with open(output_path, 'w') as f:
    json.dump(geojson, f)

size_kb = output_path.stat().st_size / 1024
print(f'✅  Exported {n_features} polygons → {output_path.name}  ({size_kb:.0f} KB)')
print(f'   Anchor: {ANCHOR_LAT}°N  {ANCHOR_LON}°E')
print(f'   Tile size: {TILE_SIZE_DEG}° (~500 m)')
print(f'\n📌 Upload this file to https://kepler.gl/demo')

In [ ]:
## 9️⃣ Quick sanity check — plot GeoJSON polygons on a map-like view

fig, ax = plt.subplots(figsize=(16, 10))

split_colors = {'train': '#1fb855', 'valid': '#2196f3', 'test': '#ff5722'}

for feat in geojson['features']:
    ring = feat['geometry']['coordinates'][0]
    lons = [c[0] for c in ring]
    lats = [c[1] for c in ring]
    sp = feat['properties']['split']
    ax.fill(lons, lats, alpha=0.35, fc=split_colors[sp], ec=split_colors[sp], lw=0.4)

ax.set_xlabel('Longitude', fontsize=11)
ax.set_ylabel('Latitude', fontsize=11)
ax.set_title('All farm field boundaries – synthetic grid layout', fontsize=13, fontweight='bold')
ax.set_aspect('equal')

# Legend
import matplotlib.patches as mpatches
handles = [mpatches.Patch(color=c, label=s, alpha=0.6) for s, c in split_colors.items()]
ax.legend(handles=handles, loc='upper right', fontsize=10)

plt.tight_layout()
plt.show()

print(f'\nTotal features: {n_features}')
print(f'Upload farm_fields_boundaries.geojson to https://kepler.gl/demo')